## <a name="ts_data_prediction">交通时序数据预测</a>

通过目标检测和OCR模型，我们可以通过交通视频分析获得交通流量的数据。接下来，我们使用不同的模型，对交通时序数据进行预测。我们暂时无法自己收集大量的交通流量信息，因此，我们使用公开的数据集进行训练。我们使用几种不同的模型对流量进行预测，包括SAEs(堆栈自动编码器，Stacked AutoEncoders），LSTM（长短时记忆，Long-Short Term Memory）和GRU（Gated Recurrent Unit，门循环网络）。

### PeMS交通流量数据集
我们使用[PeMS](http://www.dot.ca.gov/trafficops/mpr/source.html)的交通流量数据集对模型进行训练和验证，PeMS是加州交通运输部门的管理系统，其数据集由分布在城市高速交通系统中的4000多个摄像头收集而来。同时，PeMS也将历年交通流量数据进行了归档，提供超过10年的交通数据用于分析和研究。在本案例中，我们依旧使用预训练模型和最小数据集的模式，对模型进行训练。下面，我们来简单看一下PeMS数据集的结构：

In [ ]:
import pandas as pd

pems = pd.read_csv('./data/pems/train.csv')
pems.head()

In [ ]:
pems.describe()

我们主要使用时间和交通流量（每5分钟通过的车辆数量）这两列数据。

### SAEs模型

SAEs模型基于自动编码器，是一种非监督的机器学习方法。自动编码器的思想是在输入和输出之间引入一个隐含层，通过对隐含层参数的训练，使得输出数据向输入数据拟合，因此不需要对数据进行标注，是一种非监督学习。自动编码器的结构如下图所示：

![auto_encoder](./img/autoencoder.png)

假设输入$x$和输出$x'$都有$k$个特征，隐含层$a(x)$有$n$个参数，用于提取输入数据的深度特征。从输入$x$到$a(x)$成为编码，从$a(x)$到输出$x'$为解码，两个过程可以描述如下：

$$编码： a(x) = f(W_1x + b_1)$$

$$解码： x' = f(W_2a(x) + b_2)$$

其中$W_1, b_1$是编码过程的参数，$W_2, b_2$是解码过程的参数，$f$为激活函数，这里我们使用$sigmoid$函数。这样，一个单个的自动编码器就构成了一个简单的3层神经网络，可以使用反向传播对参数进行训练，目标是让输出尽量拟合输入。

SAEs则是在自动编码器的基础上，在输入和输出之间叠加多个隐含层，第一个隐含层的输入为$x$，其输出不再是$x'$，而是作为第二个隐含层的输入，以此类推，最后一个隐含层的输出为$x'$。通常情况下，编码的过程会降低数据的维度，因为我们提取的是深度的、主要的特征。这个过程在时序数据预测时，可以较好的剔除掉一些噪点数据，得到时序数据最重要的特征。SAEs的结构如下图所示：

![stacked_autoencoders](img/stacked_autoencoders.png)

在SAEs训练的过程中，是一层一层进行训练的。具体来说，训练第一个隐含层时，我们构建一个临时的输出层，与$x$维度相同，训练结束后，我们得到$a1(x)$，并将临时的输出层丢弃。因为该输出只用于训练，不需要保留，我们仅需将第一个隐含层作为输入传给一下层。接下来，将$a1(x)$作为输入传递给第二个隐含层，并构建第二个临时的输出层继续训练，以此类推。最终，得到一个维度较低的主要特征。

下面，我们来构建一个3层的SAEs模型。由于AE中的隐含层只是一个简单的参数层，可以用Dense Layer进行构建。

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout

def autoencoder(inputs, hidden, output):
    seq = Sequential()
    seq.add(Dense(hidden, input_dim=inputs, name='hidden'))
    seq.add(Activation('sigmoid'))
    seq.add(Dropout(0.2))
    seq.add(Dense(output, activation='sigmoid'))
    
    return seq

# todo more explaination of the dims
dims = [12, 400, 400, 400, 1]
ae1 = autoencoder(dims[0], dims[1], dims[-1])
ae2 = autoencoder(dims[1], dims[2], dims[-1])
ae3 = autoencoder(dims[3], dims[3], dims[-1])


saes = Sequential()
saes.add(Dense(dims[1], input_dim=dims[0], name='hidden1'))
saes.add(Activation('sigmoid'))
saes.add(Dense(dims[2], name='hidden2'))
saes.add(Activation('sigmoid'))
saes.add(Dense(dims[3], name='hidden3'))
saes.add(Activation('sigmoid'))
saes.add(Dropout(0.2))
saes.add(Dense(dims[4], activation='sigmoid'))

这样，我们就构建好了每一层AE，用于单独训练，并最终使用saes将单独训练的结果再进行训练。接下来，我们处理时序数据：

In [ ]:
from src.ts.data import process_data
lag = 12
file1 = 'data/pems/train.csv'
file2 = 'data/pems/test.csv'
X_train, y_train, _, _, _ = process_data(file1, file2, lag)

我们先单独训练每一个AE：

In [ ]:
# 单独训练每个AE层
from keras.models import Model

config = {"batch": 256, "epochs": 5}
single_aes = [ae1, ae2, ae3]

temp = X_train
for i in range(len(single_aes)):
    if i > 0:
        p = single_aes[i - 1]
        hidden_layer_model = Model(input=p.input,
                                   output=p.get_layer('hidden').output)
        temp = hidden_layer_model.predict(temp)

    m = single_aes[i]
    m.compile(loss="mse", optimizer="rmsprop", metrics=['mape'])

    m.fit(temp, y_train, batch_size=config["batch"],
          epochs=config["epochs"],
          validation_split=0.05)
    print('layer {} trained'.format(i))
    single_aes[i] = m


然后设定每一层AE的参数，并对整个SAEs模型进行训练，并保存训练结果

In [ ]:
for i in range(len(single_aes)):
    weights = single_aes[i].get_layer('hidden').get_weights()
    saes.get_layer('hidden%d' % (i + 1)).set_weights(weights)
    
# train_model(saes, X_train, y_train, 'saes', config)
saes.compile(loss="mse", optimizer="rmsprop", metrics=['mape'])

hist = saes.fit(
    X_train, y_train,
    batch_size=config["batch"],
    epochs=config["epochs"],
    validation_split=0.05)

saes.save('data/saes.h5')
df = pd.DataFrame.from_dict(hist.history)
df.to_csv('data/saes.loss.csv', encoding='utf-8', index=False)

训练完成后，查看模型的精度：

In [ ]:
df

接下来，我们使用模型预测数据：

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline

def plot_results(y_true, y_preds, names):
    d = '2016-3-4 00:00'
    x = pd.date_range(d, periods=288, freq='5min')

    fig = plt.figure()
    ax = fig.add_subplot(111)

    ax.plot(x, y_true, label='True Data')
    for name, y_pred in zip(names, y_preds):
        ax.plot(x, y_pred, label=name)

    plt.legend()
    plt.grid(True)
    plt.xlabel('Time of Day')
    plt.ylabel('Flow')

    date_format = mpl.dates.DateFormatter("%H:%M")
    ax.xaxis.set_major_formatter(date_format)
    fig.autofmt_xdate()

    plt.show()
    
    
    
lag = 12
file1 = 'data/pems/train.csv'
file2 = 'data/pems/test.csv'
_, _, X_test, y_test, scaler = process_data(file1, file2, lag)
y_test = scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(1, -1)[0]


X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1]))
predicted = saes.predict(X_test)
predicted = scaler.inverse_transform(predicted.reshape(-1, 1)).reshape(1, -1)[0]
# results = (predicted[:288])
# eva_regress(y_test, predicted)

plot_results(y_test[: 288], [predicted[:288]], ['saes'])


为了便于演示，我们训练的epoch较少，增加epoch并调整参数，可以得到更精确的模型。

### LSTM和GRU

LSTM（Long Short-Term Memory，长短时记忆）是循环神经网络（Recurrent Nerual Network，简称RNN）的一种变体，用于解决RNN中长时依赖（Long-Term Dependences）的问题。


#### RNN
RNN主要用于处理和预测时序数据。传统的神经网络结构都是从输入层到隐含层再到输出层，每一层内部的节点是不互相连接的。而RNN则有所不同，RNN是包含循环结构的神经网络，可以实现信息的持久化。人类思考问题的时，往往是根据前面的信息，来推断后续信息的可能性。例如，当我们阅读句子时，当前的词为"很"，前一个词是"天空"，那么下一个单词很有可能是"蓝"。在传统神经网络中，我们很难利用前置的"天空"这个信息来对后续的"蓝"进行预测。在RNN中，神经元会"记住"之前的信息，并对当前的输出产生影响。具体说来，RNN中一个隐含层的输入不仅包括输入层，还包括上一时刻隐含层的输出，如下图所示：

![rnn_cell](./img/rnn_cell.png)

其中，$t$代表时刻，$x$为输入层，$h$为输出层，$C$为某一时刻的状态，$A$为循环网络的主体结构。可以看到，$A$在$t-1$时刻的隐含状态$C_{t-1}$还会继续传递给$A$，与$t$时刻的输入$x_t$一起被$A$处理，产生$t$时刻的输出$h_t$。RNN中的模块$A$中的运算和变量在不同时刻是相同的，因此RNN可以看成是同一个神经网络结构被无限复制的结果。这样，我们就可以按照时间顺序将RNN展开，构成一个比较直观的神经网络，如下图所示：

![rnn_extended](./img/rnn_extended.png)

RNN在许多领域都有优异的表现。但是RNN中存在一个问题，即：难以利用时间间隔很长的信息。例如，上面的例子中，我们阅读"天空很"时，信息与要预测的时间距离很近。假如要读的句子是"在一个阴雨绵绵的午后，天空是如此"，这时，我们很大概率可以预测下面对天空的描述是"灰暗"、"阴沉"一类的词，但是这是建立在我们需要利用较早前的信息"阴雨绵绵"的基础上，因为单凭"午后"，"天空"，我们难以判断天空现在的状态。理论上，通过调整参数等手段，RNN可以解决一部分长时依赖的问题，但是在实践中却非常难，在论文[《Learning Long-Term Dependencies with Gradient Descent is Difficult》](http://ai.dinfo.unifi.it/paolo//ps/tnn-94-gradient.pdf)中有详细的分析。在复杂的时序数据场景中，对后续状态产生决定性影响的信息的时间间隔可能很长，而LSTM就是为了解决这种问题而设计的。

#### LSTM

在标准的RNN中，重复的模块A往往非常简单，例如一个$tanh$激活函数，在LSTM中，模块$A$中引入了更为复杂的结构，包括三个"门"结构：

- 遗忘门
- 输入门
- 输出门

门的结构是通过一个$sigmoid$激活函数和一个按位乘的操作来实现的，因为$sigmoid$函数输出一个0到1之间的数值，表示可以通过信息的量。当$sigmoid$输出为0时，所有信息都无法通过（被丢弃、遗忘），输出为1时，所有信息都可以通过，和逻辑电路中的门电路非常相似。LSTM中，模块的具体结构如下图所示：

![rnn_gates](img/rnn_gates.png)

我们以$t$时刻为例，$t$时刻，输入值$x_t$和上一时刻输出$h_{t-1}$传递给遗忘门，输出一个$n$维向量，在每个维度的值都是0到1之间的一个数字。该向量与上一时刻的状态$C_{t-1}$进行按位乘操作，丢弃一些不重要的信息，在向量接近0的维度上的信息会被丢弃，接近1的维度上的信息则会保留。这样，模块就"忘记"了前置信息中那些不重要的部分。

然后，模块还需要从当前的输入$x_t$中补充新的记忆。$x_t$和$h_{t-1}$先输入到一个$tanh$激活函数中，作为输入值$i$。两者再传递给输入门，输入门产生的向量再与输入值$i$进行按位乘操作，决定哪些信息得以保留。

通过遗忘门和输入门的处理之，模块可以有效的决定哪些信息应该被遗忘，哪些应该被记忆。接下来，输入门和遗忘门各自处理的结果进行+操作，得到新的状态$C_t$。

最后，$x_t$和$h_{t-1}$传递给输出门，得到的向量再与新状态$C_t$进行按位乘操作，得到最终的输出$h_t$。这样，一个完整的LSTM模块处理过程就完成了。

下面，我们通过简洁的公式，来表述这一过程（公式中为了便于表述，只保留了权重$W$，忽略偏置项$b$）：

- step1:  $f = sigmoid(W_f\cdot{[h_{t-1}, x_t]}) $，$f$为遗忘门产生的向量，与上一时刻状态$C_{t-1}$按位乘得到$f\cdot{C_{t-1}}$


- step2:  $ z = tanh(W_z\cdot{[h_{t-1}, x_t]})， i = sigmoid(W_i\cdot{[h_{t-1}, x_t]}) $，其中$z$为输入值，$i$为输入门处理得到的向量，两者进行按位乘，得到$i\cdot{z}$


- step3，前两步得到的结果进行加操作，得到新的状态：
$$ C_t = f\cdot{C_{t-1}} + i\cdot{z}  $$


- step4: $ o = sigmoid(W_o\cdot{[h_{t-1}, x_t]}) $，$o$为输出门产生的向量


- step5，新状态经过$tanh$激活，再与输出门产生的向量进行按位乘操作，得到下一时刻的输出：
$$ h_t = o\cdot{tanh(C_t)} $$


在使用LSTM预测交通流量时许数据时，抽象的理解，可以认为每一时刻的交通流量对后续时刻都会产生影响，在实际中，某一时刻发生的事件，可能对较长时间后的某一个时刻也会产生较大影响，也可能对较短时间后的某一时刻产生较大影响。这样LSTM就可以适应较为复杂的交通情况，对流量进行更为准确的预测。

#### GRU

GRU（Gated Recurrent Unit）是LSTM的一个变体，对LSTM中的门结构进行了简化。GRU中包含了更新门$z_t$和重置门$r_t$两个门结构。更新门用于控制上一时刻的信息输入到当前状态的程度，重置门用于控制忽略上一时刻状态信息的程度。同时，GRU也隐含了状态，GRU的结构如下图所示：

![gru_cell](img/gru_cell.png)

GRU的前馈过程如下所示（为了便于表述，我们仍然只保留权重$W$）：

- step1: $z_t = sigmoid(W_z\cdot{[h_{t-1}, x_t]})$，$z_t$为更新门产生的向量


- step2: $r_t = sigmoid(W_r\cdot{[h_{t-1}, x_t]})$，$r_t$为重置门产生的向量


- step3: $\tilde{h} = tanh(W\cdot{r_t \times{h_{t-1}}, x_t})$


- step4: $h_t = (1 - z_t) \times {h_{t-1}} + z_t \times {\tilde{h}}$


GRU简化了LSTM结构，参数数量更少，在一些场景下，对过拟合问题有更好的表现。

#### 训练LSTM和GRU模型

Keras对LSTM和GRU都进行了实现，并提供了简单易用的封装，我们直接使用，有兴趣的读者可以深入Keras代码，查看LSTM和GRU的实现细节。

首先，构建并训练LSTM模型：

In [ ]:
from keras.layers import Dense, Dropout, Activation
from keras.layers.recurrent import LSTM
from keras.models import Sequential

units = [12, 64, 64, 1]
lstm_model = Sequential()
lstm_model.add(LSTM(units[1], input_shape=(units[0], 1), return_sequences=True))
lstm_model.add(LSTM(units[2]))
lstm_model.add(Dropout(0.2))
lstm_model.add(Dense(units[3], activation='sigmoid'))

X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
lstm_model.compile(loss="mse", optimizer="rmsprop", metrics=['mape'])
# early = EarlyStopping(monitor='val_loss', patience=30, verbose=0, mode='auto')
hist = lstm_model.fit(X_train, y_train,
                 batch_size=config["batch"], epochs=config["epochs"],
                 validation_split=0.05)

用LSTM模型预测数据，并查看结果

In [ ]:
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
predicted = lstm_model.predict(X_test)
predicted = scaler.inverse_transform(predicted.reshape(-1, 1)).reshape(1, -1)[0]
plot_results(y_test[: 288], [predicted[:288]], ['LSTM'])

构建并训练GRU模型

In [ ]:
from keras.layers import GRU
gru_model = Sequential()
gru_model.add(GRU(units[1], input_shape=(units[0], 1), return_sequences=True))
gru_model.add(GRU(units[2]))
gru_model.add(Dropout(0.2))
gru_model.add(Dense(units[3], activation='sigmoid'))

gru_model.compile(loss="mse", optimizer="rmsprop", metrics=['mape'])
# early = EarlyStopping(monitor='val_loss', patience=30, verbose=0, mode='auto')
hist = gru_model.fit(X_train, y_train, batch_size=config["batch"],
                 epochs=config["epochs"], validation_split=0.05)

用GRU模型预测数据并查看结果：

In [ ]:
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
predicted = gru_model.predict(X_test)
predicted = scaler.inverse_transform(predicted.reshape(-1, 1)).reshape(1, -1)[0]
plot_results(y_test[: 288], [predicted[:288]], ['GRU'])